# 乱数シードによるばらつきを考慮して結果を報告する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/seed_variance.ipynb)

NLP クックブック 13-08 ／ 評価指標と統計的検定

ファインチューニングは乱数シードで結果が変わる。1 回の実行の数字ではなく、複数回の平均とばらつきで手法を比較したい。

In [1]:
import numpy as np
from scipy import stats

# 5 つのシードで学習したときのテスト F1 (例)
baseline = np.array([84.2, 85.1, 83.6, 84.9, 84.4])
proposed = np.array([85.0, 85.9, 84.1, 86.2, 85.3])

for name, x in [("baseline", baseline), ("proposed", proposed)]:
    ci = stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    print(f"{name}: {x.mean():.2f} ± {x.std(ddof=1):.2f} (標準偏差)  95% 信頼区間 [{ci[0]:.2f}, {ci[1]:.2f}]  最小 {x.min()} 最大 {x.max()}")

baseline: 84.44 ± 0.59 (標準偏差)  95% 信頼区間 [83.70, 85.18]  最小 83.6 最大 85.1
proposed: 85.30 ± 0.82 (標準偏差)  95% 信頼区間 [84.28, 86.32]  最小 84.1 最大 86.2


In [2]:
t, p = stats.ttest_ind(proposed, baseline, equal_var=False)       # Welch の t 検定
u, p_u = stats.mannwhitneyu(proposed, baseline, alternative="two-sided")   # 正規性を仮定しないノンパラメトリック検定
d = (proposed.mean() - baseline.mean()) / np.sqrt((proposed.var(ddof=1) + baseline.var(ddof=1)) / 2)
print(f"差 = {proposed.mean() - baseline.mean():+.2f}  Welch の t 検定 p = {p:.3f}  Mann-Whitney p = {p_u:.3f}  効果量 d = {d:.2f}")

# やってはいけない比較: 提案手法の最良のシードとベースラインの 1 回の結果を比べる
print(f"最良同士の差: {proposed.max() - baseline.max():+.1f} / 提案の最良 vs ベースラインの最悪: {proposed.max() - baseline.min():+.1f}")

差 = +0.86  Welch の t 検定 p = 0.098  Mann-Whitney p = 0.151  効果量 d = 1.20
最良同士の差: +1.1 / 提案の最良 vs ベースラインの最悪: +2.6


- 小さなデータセットでの BERT のファインチューニングでは、シードだけで数ポイント変わることがある。最低でも 3〜5 シードの平均と標準偏差を報告する。
- シード数が少ないと検定の検出力は低い。有意にならなくても「差がない」とは言えない。
- 両手法で同じシードを使ったからといって、対応のあるデータになるわけではない（初期値やデータ順が同じでも、手法が違えば別の乱数列になる）。